# Predictive Maintenance: EDA and Model Comparison

This notebook focuses on exploring the industrial machine maintenance dataset and comparing different machine learning models to predict potential failures. We will justify the feature selection through correlation analysis and visualize the performance of various classifiers.

In [1]:
import sys
import os
from pathlib import Path

# Add the src directory to the Python path to import our local package
sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from predictive_maintenance.config import DATASET_PATH, NUMERIC_COLUMNS, FEATURE_COLUMNS
from predictive_maintenance.data import load_dataset
from predictive_maintenance.train import train_and_compare

# Use relative path for loading data from within the notebook folder
dataset = load_dataset(Path("../") / DATASET_PATH.relative_to(DATASET_PATH.parents[1]))
dataset.head()

,timestamp,machine_id,machine_type,vibration_rms,temperature_motor,current_phase_avg,pressure_level,rpm,operating_mode,hours_since_maintenance,ambient_temp,rul_hours,failure_within_24h,failure_type,estimated_repair_cost
0,2024-01-01 00:00:00,1,CNC,0.81,49.51,5.10,23.6,860.9,idle,273.80,13.9,61.00,0,none,0
1,2024-01-01 00:03:00,1,CNC,0.75,40.58,5.30,23.6,899.6,idle,273.85,10.2,60.95,0,none,0
2,2024-01-01 00:21:00,1,CNC,0.71,49.70,NaN,21.3,862.7,idle,274.15,13.6,60.65,0,none,0
3,2024-01-01 00:45:00,1,CNC,0.76,43.04,4.79,22.6,870.4,idle,274.55,13.4,60.25,0,none,0
4,2024-01-01 00:54:00,1,CNC,0.88,41.39,4.44,22.2,881.9,idle,274.70,10.8,60.10,0,none,0


## 1. Exploratory Data Analysis (EDA)

### 1.1 Dataset Overview
We start by checking the descriptive statistics and the distribution of our target variable.

In [2]:
dataset.describe(include="all")

,timestamp,machine_id,machine_type,vibration_rms,temperature_motor,current_phase_avg,pressure_level,rpm,operating_mode,hours_since_maintenance,ambient_temp,rul_hours,failure_within_24h,failure_type,estimated_repair_cost
count,24042,24042.000000,24042,23042.000000,23208.000000,23311.000000,23118.000000,23509.000000,24042,24042.000000,24042.000000,24042.000000,24042.000000,24042,24042.000000
unique,21176,NaN,4,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,5,NaN
top,2024-01-01 00:00:00,NaN,Pump,NaN,NaN,NaN,NaN,NaN,normal,NaN,NaN,NaN,NaN,none,NaN
freq,20,NaN,6114,NaN,NaN,NaN,NaN,NaN,11632,NaN,NaN,NaN,NaN,20482,NaN
mean,NaN,10.505033,NaN,1.623667,51.404295,8.823829,59.012233,1144.849317,NaN,172.630624,12.996398,27.812510,0.148074,NaN,608.870144
std,NaN,5.746455,NaN,1.081061,12.519279,5.366391,38.723271,912.670971,NaN,150.722469,2.883994,26.393801,0.355181,NaN,1566.793887
min,NaN,1.000000,NaN,0.350000,28.000000,2.200000,10.100000,124.100000,NaN,0.000000,8.000000,0.500000,0.000000,NaN,0.000000
25%,NaN,6.000000,NaN,0.820000,42.610000,4.630000,22.700000,489.400000,NaN,42.870000,10.500000,0.500000,0.000000,NaN,0.000000
50%,NaN,10.000000,NaN,1.270000,50.060000,6.430000,46.300000,856.000000,NaN,121.610000,13.000000,22.570000,0.000000,NaN,0.000000
75%,NaN,15.000000,NaN,2.270000,59.962500,13.120000,94.700000,1676.000000,NaN,295.575000,15.500000,46.410000,0.000000,NaN,0.000000


### 1.2 Target Distribution
Understanding the balance between classes is crucial for choosing the right metrics (like F1-score over Accuracy) and for handling class imbalance during training.

In [3]:
target_counts = dataset["failure_within_24h"].value_counts().reset_index()
target_counts.columns = ["failure_within_24h", "count"]
target_counts["failure_within_24h"] = target_counts["failure_within_24h"].map({0: "No Failure", 1: "Failure"})

fig_target = px.bar(
    target_counts, 
    x="failure_within_24h", 
    y="count", 
    color="failure_within_24h",
    title="Distribution of Machine Failures within 24h",
    labels={"failure_within_24h": "Status", "count": "Number of Samples"}
)
fig_target.show()

### 1.3 Feature Distributions
Visualizing the distribution of numerical features helps identify outliers and the need for scaling.

In [4]:
fig_dist = make_subplots(rows=2, cols=3, subplot_titles=NUMERIC_COLUMNS)

for i, col in enumerate(NUMERIC_COLUMNS):
    row = i // 3 + 1
    col_idx = i % 3 + 1
    fig_dist.add_trace(
        go.Histogram(x=dataset[col], name=col),
        row=row, col=col_idx
    )

fig_dist.update_layout(height=600, width=900, title_text="Distribution of Numerical Features", showlegend=False)
fig_dist.show()

### 1.4 Correlation Matrix
Correlation analysis helps us justify which columns are useful and identify potential multicollinearity.

In [5]:
corr_matrix = dataset[NUMERIC_COLUMNS + ["failure_within_24h"]].corr()

fig_corr = px.imshow(
    corr_matrix, 
    text_auto=True, 
    aspect="auto", 
    title="Correlation Heatmap (Numerical Features & Target)",
    color_continuous_scale="RdBu_r"
)
fig_corr.show()

### 1.5 Feature vs Target Analysis
Boxplots are useful to see if certain features have significantly different distributions when a failure is imminent.

In [6]:
plot_cols = ["vibration_rms", "temperature_motor", "rul_hours"]
df_melted = dataset.melt(id_vars=["failure_within_24h"], value_vars=plot_cols)
df_melted["failure_within_24h"] = df_melted["failure_within_24h"].map({0: "Healthy", 1: "Failure"})

fig_box = px.box(
    df_melted, 
    x="failure_within_24h", 
    y="value", 
    color="failure_within_24h", 
    facet_col="variable", 
    facet_col_wrap=3,
    title="Feature Values by Failure Status",
    labels={"value": "Metric Value", "failure_within_24h": "Status"}
)
fig_box.update_yaxes(matches=None)
fig_box.show()

## 2. Model Comparison

We will now train several candidate models (e.g., Logistic Regression, Random Forest, Gradient Boosting) and compare their performance using metrics like F1-score and ROC-AUC.

In [7]:
# Run the training and comparison pipeline for binary classification
binary_summary = train_and_compare(task_name="failure_within_24h", artifact_root=Path("../artifacts"))

comparison_df = pd.read_csv("../artifacts/metrics/model_comparison_failure_within_24h.csv")
comparison_df

,model_name,f1,precision,recall,roc_auc,pr_auc
0,gradient_boosting,0.9775,0.9789,0.9761,0.9991,0.9956
1,random_forest,0.9752,0.9857,0.9649,0.9995,0.9976
2,mlp_classifier,0.9337,0.9697,0.9003,0.9974,0.9877
3,logistic_regression,0.7224,0.8456,0.6306,0.9485,0.8273


### 2.1 Visualizing Model Performance
A direct comparison helps justify why a specific model (the one with the highest F1-score) was chosen for deployment.

In [8]:
fig_perf = px.bar(
    comparison_df, 
    x="model_name", 
    y=["f1", "precision", "recall", "roc_auc"], 
    barmode="group",
    title="Model Performance Comparison (Binary Classification)",
    labels={"value": "Score", "variable": "Metric", "model_name": "Model"}
)
fig_perf.show()

### 2.2 Feature Importance (Best Model)
To justify the model's decisions, we look at which features contributed the most to the predictions.

In [9]:
importance_df = pd.read_csv("../artifacts/metrics/feature_importance_failure_within_24h.csv")

fig_imp = px.bar(
    importance_df.sort_values(by="importance_mean", ascending=True), 
    x="importance_mean", 
    y="feature", 
    orientation="h",
    title=f"Feature Importance (Permutation) - Best Model: {binary_summary['best_model_name']}",
    labels={"importance_mean": "Mean Importance (F1 Drop)", "feature": "Feature"}
)
fig_imp.show()

## 3. Multiclass Comparison
We also evaluate the performance for predicting specific failure types.

In [10]:
multiclass_summary = train_and_compare(task_name="failure_type", artifact_root=Path("../artifacts"))

multiclass_comp_df = pd.read_csv("../artifacts/metrics/model_comparison_failure_type.csv")

fig_multi = px.bar(
    multiclass_comp_df, 
    x="model_name", 
    y=["weighted_f1", "macro_f1"], 
    barmode="group",
    title="Model Performance Comparison (Multiclass Classification)",
    labels={"value": "Score", "variable": "Metric", "model_name": "Model"}
)
fig_multi.show()

/home/tonio/Workspace/m2/data_science_project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
